# Adaptive-RAG routed baseline: classifier training + evaluation (RQ1)

Paper: Jeong et al., NAACL 2024 (arXiv:2403.14403).
Code: https://github.com/starsuzi/Adaptive-RAG

Trains the t5-large query-complexity classifier and scores the routed
Adaptive-RAG system on the 6 QA test sets. These are the numbers my zero-shot
LLM router is compared against (RQ1).

Runs on a Colab GPU runtime attached from VS Code. The training cell needs
an A100 namely the authors train t5-large at batch size 32 and sequence
length 384, which OOMs on a T4's 15 GB (tried 17 Aug, died 2 steps in).
Keeping their batch size on a bigger card beats shrinking it, so the
reproduction stays faithful to the paper. Everything is re-created on a
fresh runtime: run the cells top to bottom.

In [1]:
![ -d Adaptive-RAG ] || git clone -q https://github.com/starsuzi/Adaptive-RAG.git
%cd Adaptive-RAG
# record the commit I'm working from, this goes in the thesis write-up
COMMIT = !git rev-parse HEAD
print("using commit:", COMMIT[0])

In [2]:
%%bash
set -e
# colab exports a PYTHONPATH for its own python 3.12, keep it away from the 3.8 env
unset PYTHONPATH
# upstream needs python 3.8 (torch<2 and an old transformers commit, see requirements.txt).
# colab's default python is too new for those pins, so everything runs through a
# small conda env instead. -u lets the installer rerun over an existing install.
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
bash /tmp/miniconda.sh -b -u -p /opt/miniconda
# python 3.8 from conda-forge (anaconda's default channels now need a ToS acceptance)
[ -d /opt/miniconda/envs/arag ] || \
  /opt/miniconda/bin/conda create -y -q -n arag -c conda-forge --override-channels python=3.8
/opt/miniconda/envs/arag/bin/pip install -q -r requirements.txt
/opt/miniconda/envs/arag/bin/python -c "import _jsonnet; print('env ok')"

In [3]:
%%bash
# shipped tarballs: per-strategy predictions (the classifier routes between them),
# test subsamples (ground truths), preprocessed classifier training data
tar -xzf predictions.tar.gz
tar -xzf processed_data.tar.gz
tar -xzf data.tar.gz
ls classifier/data | head -3

In [4]:
%%bash
unset PYTHONPATH
# cuda build of the pinned torch
/opt/miniconda/envs/arag/bin/pip install -q torch==1.13.1+cu117 --extra-index-url https://download.pytorch.org/whl/cu117
/opt/miniconda/envs/arag/bin/python -c "import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))"

In [5]:
%%bash
export PATH=/opt/miniconda/envs/arag/bin:$PATH
# the env ships its own libstdc++ and sqlite; without this the loader picks
# colab's older system libstdc++ and nltk's sqlite import dies on CXXABI
export LD_LIBRARY_PATH=/opt/miniconda/envs/arag/lib:${LD_LIBRARY_PATH:-}
unset PYTHONPATH
cd classifier
# two edits before training: the script pins GPU 7 (authors' server, colab has one gpu),
# and it sweeps epochs 15-35, training five times. the paper settled on epoch 25,
# so one training run is enough here.
sed -i 's/^GPU=7/GPU=0/' run/run_large_train_xl.sh
sed -i 's/^for EPOCH in 15 20 25 30 35/for EPOCH in 25/' run/run_large_train_xl.sh
bash run/run_large_train_xl.sh

In [ ]:
%%bash
export PATH=/opt/miniconda/envs/arag/bin:$PATH
export LD_LIBRARY_PATH=/opt/miniconda/envs/arag/lib:${LD_LIBRARY_PATH:-}
unset PYTHONPATH
# evaluate_final_acc.py scores the multi-hop sets with each dataset's official
# evaluator, which reads the RAW dev files. upstream's download/raw_data.sh
# pulls them from CMU, Dropbox and Google Drive, all flaky or blocked for
# unattended runs, so everything here comes from the hugging face hub instead.
[ -d official_evaluation/musique ] || bash download/official_eval.sh
pip -q install ujson pyarrow
mkdir -p raw_data/hotpotqa raw_data/2wikimultihopqa raw_data/musique .temp
H=https://huggingface.co/datasets

# hotpotqa: rebuild the raw dev json from the official hub parquet. the
# evaluator reads _id, answer and supporting_facts; checked record-for-record
# against the parquet before adopting this (7405 rows, first _id matches).
if [ ! -f raw_data/hotpotqa/hotpot_dev_distractor_v1.json ]; then
  wget -q "$H/hotpotqa/hotpot_qa/resolve/main/distractor/validation-00000-of-00001.parquet" \
    -O .temp/hotpot_dev.parquet
  python - <<'PY'
import json
import pyarrow.parquet as pq
rows = pq.read_table(".temp/hotpot_dev.parquet").to_pylist()
out = [{
    "_id": r["id"],
    "answer": r["answer"],
    "question": r["question"],
    "supporting_facts": [[t, s] for t, s in zip(r["supporting_facts"]["title"],
                                                r["supporting_facts"]["sent_id"])],
    "context": [[t, list(sents)] for t, sents in zip(r["context"]["title"],
                                                     r["context"]["sentences"])],
    "type": r["type"],
    "level": r["level"],
} for r in rows]
with open("raw_data/hotpotqa/hotpot_dev_distractor_v1.json", "w") as f:
    json.dump(out, f)
print(len(out), "hotpotqa dev records written")
PY
fi

# 2wiki: the evaluator needs answer_id per record plus the alias file, which
# the parquet mirrors drop, so take the raw files from a hub mirror of the
# original data_ids.zip (kamelliao/2wikimultihopqa).
[ -f raw_data/2wikimultihopqa/dev.json ] || \
  wget -q "$H/kamelliao/2wikimultihopqa/resolve/main/data/dev.json" \
    -O raw_data/2wikimultihopqa/dev.json
[ -f raw_data/2wikimultihopqa/id_aliases.json ] || \
  wget -q "$H/kamelliao/2wikimultihopqa/resolve/main/data/id_aliases.json" \
    -O raw_data/2wikimultihopqa/id_aliases.json

# musique: same dev split is mirrored on the hub (dgslibisey/MuSiQue, 2417
# rows, identical record structure); rebuild the jsonl from that parquet.
if [ ! -f raw_data/musique/musique_ans_v1.0_dev.jsonl ]; then
  wget -q "$H/dgslibisey/MuSiQue/resolve/refs%2Fconvert%2Fparquet/default/validation/0000.parquet" \
    -O .temp/musique_dev.parquet
  python - <<'PY'
import json
import pyarrow.parquet as pq
rows = pq.read_table(".temp/musique_dev.parquet").to_pylist()
with open("raw_data/musique/musique_ans_v1.0_dev.jsonl", "w") as f:
    for r in rows:
        f.write(json.dumps(r) + "\n")
print(len(rows), "musique dev records written")
PY
fi
ls -la raw_data/hotpotqa raw_data/2wikimultihopqa raw_data/musique


In [8]:
%%bash
export PATH=/opt/miniconda/envs/arag/bin:$PATH
# the env ships its own libstdc++ and sqlite; without this the loader picks
# colab's older system libstdc++ and nltk's sqlite import dies on CXXABI
export LD_LIBRARY_PATH=/opt/miniconda/envs/arag/lib:${LD_LIBRARY_PATH:-}
unset PYTHONPATH
# both scripts hardcode the authors' timestamped run directory. find the run I just
# trained and point them at it instead.
RESULT=$(ls -t classifier/outputs/*/model/t5-large/flan_t5_xl/epoch/*/*/*/predict/dict_id_pred_results.json | head -1)
echo "using $RESULT"
sed -i "s|^classification_result_file = .*|classification_result_file = './$RESULT'|" classifier/postprocess/predict_complexity_on_classification_results.py
python classifier/postprocess/predict_complexity_on_classification_results.py flan_t5_xl
BASE="predictions/classifier/$(echo "$RESULT" | sed 's|.*/model/||; s|/predict/.*||')/"
sed -i "s|^base_pred_path = .*|base_pred_path = './$BASE'|" evaluate_final_acc.py
python evaluate_final_acc.py

Routed EM/F1 per dataset from the cell above go into the RQ1 comparison table
(zero-shot LLM router vs trained t5-large classifier), alongside routing
accuracy once my router runs on the same test subsamples.